# Etapa 3 — Seleção de Features (Mutual Information)

Técnica única: **Mutual Information** (`mutual_info_classif` / `mutual_info_regression`),
calculada no conjunto de **treino**. Captura relações não lineares (coerente com a MLP),
lida com variáveis binárias/ordinais/contínuas e tem variante para classificação e regressão.

Lógica em `src/feature_selection.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from IPython.display import Image, Markdown

import config
from src import feature_selection as fs, preprocessing as pp

config.set_seeds()

## 1. Ranking de MI (cálculo ao vivo, rápido)

Para a tarefa binária. Troque `task` por `"multiclass"` ou `"regression"`.

In [ ]:
task = "binary"
data = pp.prepare_data(task)
ranking = fs.compute_mi_ranking(data)
ranking

## 2. Critério de corte e features selecionadas

Mantém as top-k features cuja MI acumulada atinge 90% da MI total.

In [ ]:
selected, info = fs.select_features(ranking)
print(f"Inicial: {info['n_inicial']} | Selecionadas: {info['n_selecionado']} | Descartadas: {info['n_descartado']}")
print("Critério:", info["criterio"])
print("Selecionadas:", selected)

In [ ]:
fs.plot_mi_ranking(ranking, task, info["n_selecionado"])
Image(str(config.FIGURES_DIR / f"etapa3_mi_ranking_{task}.png"))

## 3. Rankings salvos das três tarefas

In [ ]:
for t in ("binary", "multiclass", "regression"):
    sel = json.load(open(config.METRICS_DIR / f"selected_features_{t}.json", encoding="utf-8"))
    print(f"{t:11s} -> {sel['n_inicial']}->{sel['n_selecionado']} | {sel['selecionadas']}")

## 4. Comparação todas × selecionadas

Treina a MLP baseline (mesma arquitetura/seed) com todas as features e só com as
selecionadas, comparando métrica, gap treino-val e tempo de treino.

Os resultados já estão salvos em `outputs/metrics/feature_selection_comparison.csv`.
Para **recomputar** (leva alguns minutos), descomente a última linha.

In [ ]:
# Recomputar tudo (≈6 min em CPU):
# fs.run(tasks=("binary", "multiclass", "regression"), do_comparison=True)

comparison = pd.read_csv(config.METRICS_DIR / "feature_selection_comparison.csv")
comparison

In [ ]:
Image(str(config.FIGURES_DIR / "etapa3_fs_comparison.png"))

## 5. Discussão

In [ ]:
Markdown((config.METRICS_DIR / "feature_selection_discussion.md").read_text(encoding="utf-8"))